# Finetuning Qwen3.5 (image+context->table)

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

from tqdm.auto import tqdm
from unsloth import FastVisionModel

from staged_qwen3_5_scivqa.config import (
    BASE_DIR,
    COMPETITION_DATA_DIR,
    DATA_DIR,
    ENABLE_THINKING,
    LORA_CHECKPOINT_EXTRACTION,
    MODEL_ID,
    TEMPERATURE,
    TOKEN_BUDGETS,
    TOP_K,
    TOP_P,
    MIN_P,
)
from staged_qwen3_5_scivqa.preprocessing import dense_to_markdown
    PROMPT_TABLE,


In [ ]:
MAX_NEW_TOKENS = TOKEN_BUDGETS["Table"]["max_new_tokens"]
MAX_SEQUENCE_LENGTH = TOKEN_BUDGETS["Table"]["max_sequence_length"]

LORA_CHECKPOINT = LORA_CHECKPOINT_EXTRACTION

DATA_DIR.mkdir(exist_ok=True)
CATEGORY = "test"

CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

STATE_FILE = DATA_DIR / f"submission_finetuning_extraction_{CATEGORY}_state.json"
SUBMISSION_PATH = DATA_DIR / f"submission_finetuning_extraction_{CATEGORY}.json"

<a name="Data"></a>
### 🧪 Data Preparation

To convert our Sci-ImageMiner VQA data into the format required by Qwen2-VL (specifically for use with Unsloth), we need to restructure the data into a "messages" format.

The Qwen/Unsloth format expects a list of conversations where the image and the text prompt are bundled together in the user role, and the ground truth is in the assistant role, as follows:

```python
[
    { "role": "user",
    "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
    },
    { "role": "assistant",
    "content": [{"type": "text",  "text": A} ]
    },
]
```

In [ ]:
PROMPT_TEMPLATE = PROMPT_TABLE

In [ ]:

def load_test_dataset(case_dir: Path) -> list[dict]:
    samples = []
    json_files = list(case_dir.rglob("*.json"))
    pbar = tqdm(json_files, desc="Processing Subfigures")

    for json_file in pbar:
        fullpath = str(json_file)
        if (
            "content.json" in json_file.name
            or "images" not in fullpath
            or ".vscode" in fullpath
        ):
            continue

        pbar.set_description(f"Processing {json_file}")

        with open(json_file) as f:
            data = json.load(f)

        img_path = json_file.with_suffix(".jpg")
        assert img_path.exists(), f"{json_file.name} does not exist"

        # Open the full source image once
        full_img = Image.open(img_path.absolute())
        context = get_paper_context(json_file)

        # Extract bounding box info
        bboxes = data.get("bbox", {})

        # Iterate through subfigures (a, b, etc.) present in the VQA data
        for sub_key, table in data.get("data_extraction", {}).items():
            # Skip if there's no bounding box for this subfigure
            if sub_key not in bboxes:
                warnings.warn(f"Subfigure {sub_key} missing bbox in {json_file.name}")
                continue

            # Get coordinates and crop
            box = bboxes[sub_key]
            left = box["x"]
            top = box["y"]
            right = left + box["width"]
            bottom = top + box["height"]

            # Create the sub-image crop
            sub_image = full_img.crop((left, top, right, bottom))

            # Process Data Extraction associated with this sub-figure
            instruction = PROMPT_TEMPLATE.format(context=context)
            sample = convert_to_inference_conversation(
                instruction,
                sub_image,
                sample_id=data["sample_id"],
                sub_fig=sub_key,
            )
            samples.append(sample)

    return samples

Let's convert the dataset into the "correct" format for finetuning:

In [ ]:
dataset = load_test_dataset(CASE_DIR)

We look at how the conversations are structured for the first example:

In [ ]:
dataset[0]["messages"][0]["content"][1]["image"]

In [ ]:
dataset[0]["messages"]

<a name="Submission"></a>
### 📜 Creating our submission

Let's now create our submission! First, we must load the LoRA adapters we saved for inference!

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=LORA_CHECKPOINT,
    load_in_4bit=True,  # Set to False for 16bit LoRA
    max_seq_length=MAX_SEQUENCE_LENGTH,  # Must match the max_length used during training
)
FastVisionModel.for_inference(model)  # Enable for inference!

In [ ]:
state = defaultdict(dict)

if STATE_FILE.exists():
    with open(STATE_FILE) as f:
        saved_state = json.load(f)

    state = defaultdict(
        dict,
        {k: defaultdict(list, v) for k, v in saved_state.items()},
    )
    print(f"Loaded existing state from {STATE_FILE}. Resuming inference...")
else:
    state = defaultdict(dict)

In [ ]:
for sample in tqdm(dataset, desc="Running Inference"):
    meta = sample["meta"]

    existing_answers = state.get(meta["sample_id"], {}).get(meta["sub_fig"], [])
    if any(existing_answers):
        continue

    messages_content = sample["messages"][0]["content"]

    image = messages_content[1]["image"]
    instruction = messages_content[0]["text"]

    messages = [
        {
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": instruction}],
        }
    ]

    input_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=ENABLE_THINKING
    )
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        use_cache=True,
        temperature=TEMPERATURE,
        min_p=MIN_P,
        top_p=TOP_P,
        top_k=TOP_K,
    )

    # Decode only the newly generated tokens
    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1] :],
        skip_special_tokens=True,
    ).strip()

    state[meta["sample_id"]][meta["sub_fig"]] = generated

    with open(STATE_FILE, "w") as f:
        json.dump(state, f)

In [ ]:
submission = []
for sample_id, sub_figs in state.items():
    submission.append(
        {
            "sample_id": sample_id,
            "data_extraction": {
                sub_fig: dense_to_markdown(answer)
                for sub_fig, answer in sub_figs.items()
            },
        }
    )

with SUBMISSION_PATH.open("w") as f:
    json.dump(submission, f, indent=2)
print(f"Saved {len(submission)} samples to {SUBMISSION_PATH}")